# Phase 2: Feature Engineering

In this notebook, we'll perform data preprocessing and feature engineering to prepare the data for modeling.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import os


In [2]:
df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [3]:
def tenure_group(tenure):
    if tenure <= 12: return '0-12'
    elif tenure <= 24: return '13-24'
    elif tenure <= 48: return '25-48'
    elif tenure <= 60: return '49-60'
    else: return '60+'
df['TenureGroup'] = df['tenure'].apply(tenure_group)
df['AvgMonthlySpend'] = df['TotalCharges'] / df['tenure']
df['AvgMonthlySpend'] = df['AvgMonthlySpend'].fillna(0).replace([np.inf, -np.inf], 0)
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['HasMultipleServices'] = df[services].apply(lambda x: (x == 'Yes').sum(), axis=1)
contract_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df['ContractMonths'] = df['Contract'].map(contract_map)
df['Contract_Tenure_Interaction'] = df['ContractMonths'] * df['tenure']


In [4]:
target = 'Churn'
drop_cols = ['customerID']
le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn'])
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col not in drop_cols + [target]]
df_final = pd.get_dummies(df.drop(columns=drop_cols), columns=categorical_cols, drop_first=True)


In [5]:
X = df_final.drop(columns=[target])
y = df_final[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)


In [6]:
np.save('../data/X_train.npy', X_train_scaled.values)
np.save('../data/X_test.npy', X_test_scaled.values)
np.save('../data/y_train.npy', y_train.values)
np.save('../data/y_test.npy', y_test.values)
with open('../data/feature_names.pkl', 'wb') as f: pickle.dump(X_train.columns.tolist(), f)
with open('../data/scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
